In [52]:
import netCDF4
from netCDF4 import Dataset
import sys, os

import numpy as np
import pandas as pd 
from scipy.stats import wasserstein_distance_nd
import scipy
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
import xarray as xr

In [53]:
rootgrp_1 = Dataset("/home/mikhail/ML/Diploma_ocean/Data/Data_Mikhail_lab\
/vit_finetuned_lead30d.nc", "r", format="NETCDF4")
rootgrp_1.variables

{'depth': <class 'netCDF4.Variable'>
 float32 depth(depth)
     long_name: Depth
     standard_name: depth
     units: m
     positive: down
     axis: Z
 unlimited dimensions: 
 current shape = (1,)
 filling on, default _FillValue of 9.969209968386869e+36 used,
 'time': <class 'netCDF4.Variable'>
 float64 time(time)
     long_name: Time
     standard_name: time
     units: seconds since 1970-01-01 00:00:00
     calendar: gregorian
     axis: T
 unlimited dimensions: 
 current shape = (6,)
 filling on, default _FillValue of 9.969209968386869e+36 used,
 'latitude': <class 'netCDF4.Variable'>
 float32 latitude(latitude)
     long_name: Latitude
     standard_name: latitude
     units: degrees_north
     axis: Y
 unlimited dimensions: 
 current shape = (92,)
 filling on, default _FillValue of 9.969209968386869e+36 used,
 'longitude': <class 'netCDF4.Variable'>
 float32 longitude(longitude)
     long_name: Longitude
     standard_name: longitude
     units: degrees_east
     axis: X
 unlim

In [54]:
rootgrp_1 = Dataset("/home/mikhail/ML/Diploma_ocean/Models/Models_data/out_file.nc", "r", format="NETCDF4")
rootgrp_2 = Dataset("/home/mikhail/ML/Diploma_ocean/Data/Glorys/Glo_mae_examp.nc", "r", format="NETCDF4")
# rootgrp_2 = Dataset("/home/mikhail/ML/Diploma_ocean/Models/Models_data/out_file.nc", "r", format="NETCDF4")

# print(rootgrp_2.variables['vo'])
# print(rootgrp_1.variables['vo'])

def mae_def(day, rootgrp_1, rootgrp_2):
    data_1 = rootgrp_1.variables['uo'][day,0,:,:]
    data_2 = rootgrp_2.variables['uo'][day+1,0,:,:]
    data_3 = rootgrp_1.variables['vo'][day,0,:,:]
    data_4 = rootgrp_2.variables['vo'][day+1,0,:,:]
    
    data_1 = data_1.filled(0)
    data_2 = data_2.filled(0)
    data_3 = data_3.filled(0)
    data_4 = data_4.filled(0)
    
    mae_uo = mean_absolute_error(data_1, data_2)
    mae_vo = mean_absolute_error(data_3, data_4)
    return mae_uo, mae_vo

# print(mae_def(5, rootgrp_1, rootgrp_2))

In [55]:
rootgrp_1 = Dataset("/home/mikhail/ML/Diploma_ocean/Models/Models_data/out_file.nc", "r", format="NETCDF4")
rootgrp_2 = Dataset("/home/mikhail/ML/Diploma_ocean/Data/Glorys/Glo_mae_examp.nc", "r", format="NETCDF4")
# rootgrp_2 = Dataset("/home/mikhail/ML/Diploma_ocean/Models/Models_data/out_file.nc", "r", format="NETCDF4")

# print(rootgrp_1.variables['uo'][0, 0, 1500, 2550])

# print(rootgrp_2.variables['vo'])
# print(rootgrp_1.variables['vo'])

def mae_all_def(day, rootgrp_1, rootgrp_2):
    data_1 = rootgrp_1.variables['uo'][day,0,:,:]
    data_2 = rootgrp_2.variables['uo'][day+1,0,:,:]
    
    data_3 = rootgrp_1.variables['vo'][day,0,:,:]
    data_4 = rootgrp_2.variables['vo'][day+1,0,:,:]

    data_1 = data_1.filled(0)
    data_2 = data_2.filled(0)
    data_3 = data_3.filled(0)
    data_4 = data_4.filled(0)

    data_uo = data_2 - data_1
    data_vo = data_4 - data_3

    data_fin = np.sqrt(data_uo ** 2 + data_vo ** 2)
    
    
    return np.mean(data_fin), data_fin

output, data_fin = mae_all_def(5, rootgrp_1, rootgrp_2)

hist_mtx = []
for i in range(len(data_fin)):
    hist_mtx.append(data_fin[i])
    
# print(hist_mtx)
# plt.hist(hist_mtx, bins=20, range=[0, 0.5])

counts, bins = np.histogram(hist_mtx, bins=20, range=[0, 0.5])
# plt.stairs(counts, bins)
# print(counts, bins)


# print(data_fin)

In [56]:
# def MAE_polygon(link_1, link_2, init_a, out_a, flg=0):
#     rootgrp_1 = Dataset(link_1, "r", format="NETCDF4")

#     coord_1_x = np.array(rootgrp_1.variables['longitude'][init_a:out_a].tolist())
#     coord_1_y = np.array(rootgrp_1.variables['latitude'][init_a:out_a].tolist())
#     if flg == 0:
#         rootgrp_2 = Dataset(link_2, "r", format="NETCDF4")
#         coord_2_x = np.array(rootgrp_2.variables['longitude'][init_a:out_a].tolist())
#         coord_2_y = np.array(rootgrp_2.variables['latitude'][init_a:out_a].tolist())
#     else:
#         file = open(link_2)
#         coord_2_x = []
#         coord_2_y = []
#         for line in file:
#             # print(line)
#             line_str = line.split()
#             coord_2_x.append(float(line_str[1]))
#             coord_2_y.append(float(line_str[0]))
            
#     coord_2_x = np.resize(np.array(coord_2_x), 500)
#     coord_2_y = np.resize(np.array(coord_2_y), 500)

#     coord_1 = []
#     coord_2 = []
#     coord_3 = []
#     for i in range(500):
#         coord_1.append([coord_1_x[i], coord_1_y[i]])
#         coord_2.append([coord_2_x[i], coord_2_y[i]])
        
#     poly_coords_1 = coord_1
#     poly_1 = Polygon(poly_coords_1, closed=True, facecolor='lightblue',\ 
#                edgecolor='blue', alpha=0.6)
    
#     poly_coords_2 = coord_2
#     poly_2 = Polygon(poly_coords_2, closed=True, facecolor='lightblue',\ 
#                edgecolor='blue', alpha=0.6)
#     for i in range (1000):
        
    

In [57]:
def Euclid(link_1, link_2, init_a, out_a, flg=0):
    num_points = out_a - init_a
    rootgrp_1 = Dataset(link_1, "r", format="NETCDF4")

    coord_1_x = np.array(rootgrp_1.variables['longitude'][init_a:out_a].tolist())
    coord_1_y = np.array(rootgrp_1.variables['latitude'][init_a:out_a].tolist())
    
    if flg == 0:
        rootgrp_2 = Dataset(link_2, "r", format="NETCDF4")
        coord_2_x = np.array(rootgrp_2.variables['longitude'][init_a:out_a].tolist())
        coord_2_y = np.array(rootgrp_2.variables['latitude'][init_a:out_a].tolist())
    
    else:
        file = open(link_2)
        coord_2_x = []
        coord_2_y = []
        for line in file:
            # print(line)
            line_str = line.split()
            coord_2_x.append(float(line_str[1]))
            coord_2_y.append(float(line_str[0]))
    coord_2_x = np.resize(np.array(coord_2_x), num_points)
    coord_2_y = np.resize(np.array(coord_2_y), num_points)
        
    # Euclid = np.mean(np.sqrt(np.square(np.abs(coord_2_x - coord_1_x)) +\
    #                       np.square(np.abs(coord_2_y - coord_1_y))))
    coord_1 = []
    coord_2 = []
    for i in range(num_points):
        coord_1.append([coord_1_x[i], coord_1_y[i]])
        coord_2.append([coord_2_x[i], coord_2_y[i]])
        
    Euclid_mtx = scipy.spatial.distance.cdist(coord_1, coord_2, 'euclidean')
    
    Euclid = sum(sum(Euclid_mtx)) / (len(coord_1) * len(coord_2))
    return Euclid

In [58]:
def Euclid_km(link_1, link_2, init_a, out_a, flg=0):
    num_points = out_a - init_a
    rootgrp_1 = Dataset(link_1, "r", format="NETCDF4")

    coord_1_x = np.array(rootgrp_1.variables['longitude'][init_a:out_a].tolist())
    coord_1_y = np.array(rootgrp_1.variables['latitude'][init_a:out_a].tolist())
    
    if flg == 0:
        rootgrp_2 = Dataset(link_2, "r", format="NETCDF4")
        coord_2_x = np.array(rootgrp_2.variables['longitude'][init_a:out_a].tolist())
        coord_2_y = np.array(rootgrp_2.variables['latitude'][init_a:out_a].tolist())
    
    else:
        file = open(link_2)
        coord_2_x = []
        coord_2_y = []
        for line in file:
            # print(line)
            line_str = line.split()
            coord_2_x.append(float(line_str[1]))
            coord_2_y.append(float(line_str[0]))
    coord_2_x = np.resize(np.array(coord_2_x), num_points)
    coord_2_y = np.resize(np.array(coord_2_y), num_points)
        
    # Euclid = np.mean(np.sqrt(np.square(np.abs(coord_2_x - coord_1_x)) +\
    #                       np.square(np.abs(coord_2_y - coord_1_y))))
    coord_1 = []
    coord_2 = []
    coord_3 = []
    coord_4 = []
    for i in range(num_points):
        coord_1.append([coord_1_x[i], coord_1_y[i]])
        coord_2.append([coord_2_x[i], coord_2_y[i]])
        
        coord_3.append([(coord_1_x[i]-37)*80.07, (coord_1_y[i]-44)*111.1])
        coord_4.append([(coord_2_x[i]-37)*80.07, (coord_2_y[i]-44)*111.1])
        
    Euclid_mtx = scipy.spatial.distance.cdist(coord_3, coord_4, 'euclidean')
    
    Euclid_km = sum(sum(Euclid_mtx)) / (len(coord_1) * len(coord_2))
    return Euclid_km

In [59]:
# def wass_metric_km(link_1, link_2, init_a, out_a, flg=0):
#     num_points = out_a - init_a
#     rootgrp_1 = Dataset(link_1, "r", format="NETCDF4")

#     coord_1_x = np.array(rootgrp_1.variables['longitude'][init_a:out_a].tolist())
#     coord_1_y = np.array(rootgrp_1.variables['latitude'][init_a:out_a].tolist())
#     if flg == 0:
#         rootgrp_2 = Dataset(link_2, "r", format="NETCDF4")
#         coord_2_x = np.array(rootgrp_2.variables['longitude'][init_a:out_a].tolist())
#         coord_2_y = np.array(rootgrp_2.variables['latitude'][init_a:out_a].tolist())
#     else:
#         file = open(link_2)
#         coord_2_x = []
#         coord_2_y = []
#         for line in file:
#             # print(line)
#             line_str = line.split()
#             coord_2_x.append(float(line_str[1]))
#             coord_2_y.append(float(line_str[0]))
#     coord_2_x = np.resize(np.array(coord_2_x), num_points)
#     coord_2_y = np.resize(np.array(coord_2_y), num_points)

#     coord_1 = []
#     coord_2 = []
#     coord_3 = []
#     coord_4 = []
#     for i in range(num_points):
#         coord_1.append([coord_1_x[i], coord_1_y[i]])
#         coord_2.append([coord_2_x[i], coord_2_y[i]])
        
#         coord_3.append([(coord_2_x[i]-coord_1_x[i])*80.07, (coord_2_y[i]-coord_1_y[i])*111.1])
#         coord_4.append([0, 0])

            
#     wass_metric = wasserstein_distance_nd(coord_3, coord_4)
    
#     return wass_metric

In [60]:
def wass_metric_km(link_1, link_2, init_a, out_a, flg=0):
    num_points = out_a - init_a
    rootgrp_1 = Dataset(link_1, "r", format="NETCDF4")

    coord_1_x = np.array(rootgrp_1.variables['longitude'][init_a:out_a].tolist())
    coord_1_y = np.array(rootgrp_1.variables['latitude'][init_a:out_a].tolist())
    if flg == 0:
        rootgrp_2 = Dataset(link_2, "r", format="NETCDF4")
        coord_2_x = np.array(rootgrp_2.variables['longitude'][init_a:out_a].tolist())
        coord_2_y = np.array(rootgrp_2.variables['latitude'][init_a:out_a].tolist())
    else:
        file = open(link_2)
        coord_2_x = []
        coord_2_y = []
        for line in file:
            # print(line)
            line_str = line.split()
            coord_2_x.append(float(line_str[1]))
            coord_2_y.append(float(line_str[0]))
    coord_2_x = np.resize(np.array(coord_2_x), num_points)
    coord_2_y = np.resize(np.array(coord_2_y), num_points)

    coord_1 = []
    coord_2 = []
    coord_3 = []
    coord_4 = []
    for i in range(num_points):
        coord_1.append([coord_1_x[i], coord_1_y[i]])
        coord_2.append([coord_2_x[i], coord_2_y[i]])
        
        coord_3.append([(coord_1_x[i]-37)*80.07, (coord_1_y[i]-44)*111.1])
        coord_4.append([(coord_2_x[i]-37)*80.07, (coord_2_y[i]-44)*111.1])

            
    wass_metric_km = wasserstein_distance_nd(coord_3, coord_4)
    
    return wass_metric_km

In [61]:
def wass_metric(link_1, link_2, init_a, out_a, flg=0):
    num_points = out_a - init_a
    rootgrp_1 = Dataset(link_1, "r", format="NETCDF4")

    coord_1_x = np.array(rootgrp_1.variables['longitude'][init_a:out_a].tolist())
    coord_1_y = np.array(rootgrp_1.variables['latitude'][init_a:out_a].tolist())
    if flg == 0:
        rootgrp_2 = Dataset(link_2, "r", format="NETCDF4")
        coord_2_x = np.array(rootgrp_2.variables['longitude'][init_a:out_a].tolist())
        coord_2_y = np.array(rootgrp_2.variables['latitude'][init_a:out_a].tolist())
    else:
        file = open(link_2)
        coord_2_x = []
        coord_2_y = []
        for line in file:
            # print(line)
            line_str = line.split()
            coord_2_x.append(float(line_str[1]))
            coord_2_y.append(float(line_str[0]))
    coord_2_x = np.resize(np.array(coord_2_x), num_points)
    coord_2_y = np.resize(np.array(coord_2_y), num_points)

    coord_1 = []
    coord_2 = []
    for i in range(num_points):
        coord_1.append([coord_1_x[i], coord_1_y[i]])
        coord_2.append([coord_2_x[i], coord_2_y[i]])

            
    wass_metric = wasserstein_distance_nd(coord_1, coord_2)
    
    return wass_metric

In [62]:
def claster_metric(link_1, link_2, init_a, out_a, flg=0):
    num_points = out_a - init_a
    rootgrp_1 = Dataset(link_1, "r", format="NETCDF4")

    coord_1_x = np.array(rootgrp_1.variables['longitude'][init_a:out_a].tolist())
    coord_1_y = np.array(rootgrp_1.variables['latitude'][init_a:out_a].tolist())
    if flg == 0:
        rootgrp_2 = Dataset(link_2, "r", format="NETCDF4")
        coord_2_x = np.array(rootgrp_2.variables['longitude'][init_a:out_a].tolist())
        coord_2_y = np.array(rootgrp_2.variables['latitude'][init_a:out_a].tolist())
    else:
        file = open(link_2)
        coord_2_x = []
        coord_2_y = []
        for line in file:
            # print(line)
            line_str = line.split()
            coord_2_x.append(float(line_str[1]))
            coord_2_y.append(float(line_str[0]))
            
    coord_2_x = np.resize(np.array(coord_2_x), num_points)
    coord_2_y = np.resize(np.array(coord_2_y), num_points)

    coord_1 = []
    coord_2 = []
    coord_3 = []
    for i in range(num_points):
        coord_1.append([coord_1_x[i], coord_1_y[i]])
        coord_2.append([coord_2_x[i], coord_2_y[i]])
        
        coord_3.append([coord_1_x[i], coord_1_y[i]])
        coord_3.append([coord_2_x[i], coord_2_y[i]])

    var_1 = np.var(coord_1)
    var_2 = np.var(coord_2)
    
    metric = np.max([var_1, var_2])/np.var(coord_3)
    
    return metric


In [68]:
my_path_gn = os.path.abspath("./output")

link_GD = my_path_gn + "/test_output_GloDat_BS.nc"
link_GD_wW = my_path_gn + "/test_output_GloDat_BS_wW.nc"

link_ND1 = my_path_gn + "/test_output_NData1.nc"
link_ND1_wW = my_path_gn + "/test_output_NData1_wW.nc"

link_ND2 = my_path_gn + "/test_output_NData2.nc"

link_GOR = my_path_gn + "/test_output_GOR.nc"
link_GOR_wW = my_path_gn + "/test_output_GOR_wW.nc"

link_GOAF = my_path_gn + "/test_output_GOAF.nc"
link_GOAF_wW = my_path_gn + "/test_output_GOAF_wW.nc"

link_mik_v1 = my_path_gn + "/test_output_Mik_dat_wW.nc"


link_SD_1_09 = '/home/mikhail/ML/Diploma_ocean/Data/Data_real/Data_1_09.txt'
link_SD_30_08 = '/home/mikhail/ML/Diploma_ocean/Data/Data_real/Data_30_08.txt'
link_SD_29_08 = '/home/mikhail/ML/Diploma_ocean/Data/Data_real/Data_29_08.txt'

link_in = link_mik_v1

num_model = 500
del_mod = 145

num_in = 2*500
print('Euclid_29_08', Euclid(link_in, link_SD_29_08, del_mod+num_in, del_mod+num_in+num_model, flg=1))
print('Euclid_km_29_08', Euclid_km(link_in, link_SD_29_08, del_mod+num_in, del_mod+num_in+num_model, flg=1))
print('Wass_29_08', wass_metric(link_in, link_SD_29_08, del_mod+num_in, del_mod+num_in+num_model, flg=1))
print('Wass_29_08_km', wass_metric_km(link_in, link_SD_29_08, del_mod+num_in, del_mod+num_in+num_model, flg=1))


print('\n')
num_in = 8 * num_model
print('Euclid_30_08', Euclid(link_in, link_SD_30_08, del_mod+num_in, del_mod+num_in+num_model, flg=1))
print('Euclid_km_30_08', Euclid_km(link_in, link_SD_30_08, del_mod+num_in, del_mod+num_in+num_model, flg=1))
print('Wass_30_08', wass_metric(link_in, link_SD_30_08, del_mod+num_in, del_mod+num_in+num_model, flg=1))
print('Wass_30_08_km', wass_metric_km(link_in, link_SD_30_08, del_mod+num_in, del_mod+num_in+num_model, flg=1))


print('\n')
num_in = 24 * num_model
print('Euclid_1_09', Euclid(link_in, link_SD_1_09, del_mod+num_in, del_mod+num_in+num_model, flg=1))
print('Euclid_km_1_09', Euclid_km(link_in, link_SD_1_09, del_mod+num_in, del_mod+num_in+num_model, flg=1))
print('Wass_1_09', wass_metric(link_in, link_SD_1_09, del_mod+num_in, del_mod+num_in+num_model, flg=1))
print('Wass_1_09_km', wass_metric_km(link_in, link_SD_1_09, del_mod+num_in, del_mod+num_in+num_model, flg=1))


Euclid_29_08 0.08768430723595977
Euclid_km_29_08 8.169417698667898
Wass_29_08 0.05224716142093325
Wass_29_08_km 5.3526805481909845


Euclid_30_08 0.3782059363818036
Euclid_km_30_08 37.03882418680556
Wass_30_08 0.37538665164576285
Wass_30_08_km 36.80777121962287


Euclid_1_09 0.8901947054230593
Euclid_km_1_09 78.67230032065953
Wass_1_09 0.8889318328117526
Wass_1_09_km 78.53025679367286


## Метрики ##
### GOAF ###
+ Wind
+ Desc

Euclid 0.5630846820026391

Wass 0.5599570630032862

Clast 1.073079404961659

+ Without wind
+ Desc

Euclid 0.4897501947606939

Wass 0.4876591815600504

Clast 1.0724627832038582



### Global ocean reanal ###

+ Wind
+ Desc

Euclid 0.7472723186378314

Wass 0.7439819056703831

Clast 1.1197655066358587

+ Without wind
+ Desc

Euclid 0.7514194645011393

Wass 0.7499803156072781

Clast 1.1228115915374317


if True:
    file_list = '/home/mikhail/ML/Diploma_ocean/Data/Glorys/Glob_ocean_reanal/Glorys_glob_ocean_reanal.nc'
                

### Glorys data Reanal BS + wind ###
+ Wind
+ Desc

Euclid 0.7216159976998427

Wass 0.7148834542017704

Clast 1.1014142942957155

+ Without wind
+ Desc

Euclid 0.785288772620642

Wass 0.783283141185561

Clast 1.1227174907832391

point_line_spill

500 точек

Data:
    file_list = 
    
            'Data/Glorys/BS_hourly/20250828_h-CMCC--RFVL-BSeas8-BLK-b20250909_an-sv14.00.nc',
    
            'Data/Glorys/BS_hourly/20250829_h-CMCC--RFVL-BSeas8-BLK-b20250909_an-sv14.00.nc',
            
             'Data/Glorys/BS_hourly/20250830_h-CMCC--RFVL-BSeas8-BLK-b20250909_an-sv14.00.nc',
             
             'Data/Glorys/BS_hourly/20250831_h-CMCC--RFVL-BSeas8-BLK-b20250909_an-sv14.00.nc',
             
             'Data/Glorys/BS_hourly/20250901_h-CMCC--RFVL-BSeas8-BLK-b20250916_an-sv14.00.nc',
             
             'Data/Glorys/BS_hourly/20250902_h-CMCC--RFVL-BSeas8-BLK-b20250916_an-sv14.00.nc'




### Nural WenHai data 1 ###

+ Wind
+ Desc

Euclid 0.6499915507654364

Wass 0.647110828393125

Clast 1.1004026057420442


+ Without wind
+ Desc

Euclid 0.6363324080246168

Wass 0.6332012135906101

Clast 1.1020331488714281

+ Data
+ Desc

point_line_spill

500 точек

if True:
    file_list = '/home/mikhail/ML/Diploma_ocean/Models/Models_data/out_file.nc'



### Nural WenHai data 2 + wind ###

Euclid 0.6499915507654364

Wass 0.647110828393125

Clast 1.1004026057420442

point_line_spill

500 точек

if True:
    file_list = '/home/mikhail/ML/Diploma_ocean/Models/Models_data_2/out_file.nc'

## Варьирование точек GOAF ##

num      Euclid_1_09
20     0.6338171864738932
50     0.6484507702183125
80     0.5988726427920164
110    0.5829926785610076
140    0.569512202664199
170    0.5566174623344983
200    0.5437156903276674
230    0.533297602208763
260    0.522385510451994
290    0.5127282498750227
320    0.5030507612422549
350    0.49309561971775484
380    0.4826595404345962
410    0.4846167301521647
440    0.49300708122753106
470    0.49802512928986903
500    0.5008479235009663
530    0.5019574314066123
560    0.5017527035396657
590    0.5003573033901416
620    0.4985330469808818
650    0.49601471326097757